# Анализ модели DreamRecFMText

Архитектура DreamRecText (замороженные TEM-эмбеддинги, SASRec-энкодер, concat-денойзер
`Linear(3d, d)`), генеративный процесс — Flow Matching: сеть предсказывает скорость
`u = z₁ − z₀`, сэмплинг — Euler ODE от шума `z₀` к эмбеддингу товара `z₁`.

Модель собирается из `configs/dreamrec_fm_text.yaml` через реестр `builders.MODELS`, веса
берутся из чекпойнта. Секция 1 воспроизводит протокол тренера: пересчитанный val должен
совпасть с метриками в чекпойнте.

In [ ]:
import os, random, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)                          # пути в yaml относительны корня проекта
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from builders import load_model_config
from data import build_dataloaders
from evaluate import evaluate, evaluate_retrievals, hit_ratio, ndcg
from models.dreamrec import retrieve
from train import seeded

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
print(f'Корень проекта: {ROOT}')

In [ ]:
# ════════════════════════════════════════════════════════════
# КОНФИГ
# ════════════════════════════════════════════════════════════
CONFIG_PATH = 'configs/dreamrec_fm_text.yaml'
CKPT_DIR    = 'checkpoints/dreamrec_fm_text'

N_SAMPLES = 1000   # примеров для усреднения траектории сэмплинга
N_RUNS    = 5      # прогонов сэмплинга для mean ± std
SEED      = 42     # EVAL_SEED тренера: с этим сидом метрики совпадают с логом обучения

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Device: {device}')

In [ ]:
# ════════════════════════════════════════════════════════════
# Модель собирается по конфигу, веса — из чекпойнта
# ════════════════════════════════════════════════════════════
spec, cfg = load_model_config(CONFIG_PATH)
model = spec.build(cfg).to(device).eval()

ckpt_path = Path(CKPT_DIR) / 'best.pt'
if not ckpt_path.exists():
    ckpt_path = Path(CKPT_DIR) / 'latest.pt'
ckpt = torch.load(ckpt_path, map_location='cpu', mmap=True)
model.load_state_dict(ckpt['model_state'])      # строго: архитектура из yaml должна совпасть

NUM_STEPS  = cfg.eval.num_sample_steps          # шагов сэмплинга — как при обучении
W_CFG      = cfg.model.w_cfg
BATCH_SIZE = cfg.data.batch_size                # от него зависит нарезка потока шума
N_ITEMS    = model.n_items
W_ITEMS    = model.items.table.detach()         # (N_items, d_emb) — таблица для ретривала
RETRIEVAL_ROW = {'dot': 'Dot Product', 'cosine': 'Cosine Sim', 'l2': 'L2 Distance'}[model.retrieval_metric]

print(f'{ckpt_path} | эпоха {ckpt["epoch"]}')
print(f'model_cfg: {ckpt.get("model_cfg") or "не записан (чекпойнт старше model.config())"}')
print(f'метрики в чекпойнте: {ckpt["metrics"]}')
print(f'{model.param_summary()}')
print(f'шагов сэмплинга {NUM_STEPS} | w_cfg {W_CFG} | ретривал при обучении: {RETRIEVAL_ROW}')

In [ ]:
loaders = build_dataloaders(data_dir=cfg.data.data_dir, batch_size=BATCH_SIZE, num_workers=0,
                            seq_len=cfg.data.max_seq_len, splits=('train', 'val', 'test'))
for split, loader in loaders.items():
    print(f'{split:5s}: {len(loader.dataset):,} sequences')

## 1. Метрики по протоколу тренера

Ровно как финальный тест в `train.py`: `evaluate()` под сидом 42, с `num_sample_steps`, `w_cfg`
и `batch_size` из конфига. Пересчитанный val должен совпасть с метриками в чекпойнте — это
проверка того, что модель собрана и загружена правильно.

In [ ]:
# ════════════════════════════════════════════════════════════
# Метрики по протоколу тренера
# ════════════════════════════════════════════════════════════
def protocol_eval(split):
    with seeded(SEED, device):
        return evaluate(model=model, dataloader=loaders[split], top_k_values=cfg.eval.top_k,
                        num_sample_steps=NUM_STEPS, w_cfg=W_CFG, device=device, show_progress=False)

val_metrics, test_metrics = protocol_eval('val'), protocol_eval('test')
print(pd.DataFrame({'val (чекпойнт)': ckpt['metrics'], 'val (пересчёт)': val_metrics,
                    'test': test_metrics}).T.to_string(float_format=lambda x: f'{x:.4f}'))

diff = max(abs(val_metrics[k] - ckpt['metrics'][k]) for k in ckpt['metrics'])
verdict = 'модель загружена корректно' if diff < 1e-3 else 'НЕ совпадает: проверь CONFIG_PATH и CKPT_DIR'
print(f'\nmax |пересчёт − чекпойнт| = {diff:.1e} → {verdict}')

## 2. Все ретривалы и разброс по сидам сэмплинга

`evaluate_retrievals()` считает dot / cosine / L2 по одному и тому же oracle, поэтому строки
сравнимы между собой; `Random` — эмпирический бейзлайн. Разброс здесь только от шума сэмплинга:
различия между обучениями с разными сидами он не показывает.

То же самое из командной строки:
`python src/evaluate.py --config configs/dreamrec_fm_text.yaml --checkpoint checkpoints/dreamrec_fm_text/best.pt`

In [ ]:
# ════════════════════════════════════════════════════════════
# Вспомогательные функции
# ════════════════════════════════════════════════════════════
METRICS = [f'{p}@{k}' for k in cfg.eval.top_k for p in ('HR', 'NDCG')]
MAX_K   = max(cfg.eval.top_k)


def metrics_row(rankings, targets):
    """(N, MAX_K) предсказанных индексов + (N,) таргеты → {метрика: значение}."""
    return {f'{prefix}@{k}': fn(rankings, targets, k)
            for k in cfg.eval.top_k for prefix, fn in (('HR', hit_ratio), ('NDCG', ndcg))}


def mean_std_table(runs, rows=None):
    """runs: список {строка: {метрика: значение}} → таблица 'mean ± std'."""
    rows = rows or list(runs[0])
    table = [{'Retrieval': name,
              **{m: f'{np.mean([r[name][m] for r in runs]):.4f} ± {np.std([r[name][m] for r in runs]):.4f}'
                 for m in METRICS}}
             for name in rows]
    return pd.DataFrame(table).set_index('Retrieval')


def retrieval_runs(split, n_runs=N_RUNS):
    """n_runs прогонов сэмплинга; все ретривалы считаются по одному и тому же oracle."""
    runs = []
    for i in range(n_runs):
        print(f'  {split} run {i + 1}/{n_runs}...', end='\r')
        with seeded(SEED + 100 * i, device):
            runs.append(evaluate_retrievals(model, loaders[split], top_k_values=cfg.eval.top_k,
                                            num_sample_steps=NUM_STEPS, w_cfg=W_CFG, device=device))
    print()
    return runs


@torch.no_grad()
def oracle_for(history, mask):
    """История → oracle-эмбеддинг тем же сэмплером, которым считаются метрики."""
    return model.sample(model.encode_history(history, mask), NUM_STEPS, W_CFG)


# Прогоны переиспользуются в shuffle-тесте ниже, поэтому считаем их один раз
RETRIEVAL_RUNS = {split: retrieval_runs(split) for split in ('val', 'test')}

for split, runs in RETRIEVAL_RUNS.items():
    print(f'=== {split.upper()} ({N_RUNS} сидов сэмплинга) ===')
    print(mean_std_table(runs).to_string())
    print()

## 3. Траектория ODE

Один прогон Euler ODE с записью на каждом шаге: расстояние до таргета и до случайного товара,
норма z_τ. Шаги те же, что в `processes.FlowMatching.sample`, поэтому траектория соответствует
сэмплеру, которым считались метрики выше.

In [ ]:
# Ориентиры для графика нормы: E‖шум‖ = sqrt(d_emb) и средняя норма таргета в train
_train_df  = pd.read_pickle(Path(cfg.data.data_dir) / 'train_data.df')
_freq      = pd.Series(0, index=range(N_ITEMS)); _freq.update(_train_df['next'].value_counts())
FREQ_VALS  = _freq.values.astype(float)
ITEM_NORMS = W_ITEMS.norm(dim=-1).cpu().numpy()
_seen      = FREQ_VALS > 0
TRAIN_TARGET_NORM = float(np.average(ITEM_NORMS[_seen], weights=FREQ_VALS[_seen]))
NOISE_NORM = float(np.sqrt(model.d_emb))
print(f'E[||e_target||] по train = {TRAIN_TARGET_NORM:.3f}')
print(f'd_emb = {model.d_emb}, E[||noise||] = sqrt(d_emb) = {NOISE_NORM:.2f}')


def _tracker(target_emb, random_emb):
    """Накопитель расстояний до таргета и до случайного товара на каждом шаге."""
    track = {k: [] for k in ['l2_target', 'l2_random', 'cos_target', 'cos_random', 'norm_x']}

    def record(x):
        for name, ref in [('target', target_emb), ('random', random_emb)]:
            track[f'l2_{name}'].append((x - ref).norm(dim=-1).mean().item())
            track[f'cos_{name}'].append(F.cosine_similarity(x, ref, dim=-1).mean().item())
        track['norm_x'].append(x.norm(dim=-1).mean().item())

    return track, record


@torch.no_grad()
def sample_with_tracking(h, target_emb, num_steps=None, w_cfg=None):
    """Тот же Euler ODE, что processes.FlowMatching.sample, но с записью расстояний на каждом шаге."""
    n  = num_steps or model.process.DEFAULT_STEPS
    dt = 1.0 / n
    w  = W_CFG if w_cfg is None else w_cfg

    random_emb    = W_ITEMS[torch.randint(0, N_ITEMS, (h.shape[0],), device=device)]
    track, record = _tracker(target_emb, random_emb)

    z = torch.randn_like(h)
    record(z)
    for i in range(n):
        tau = torch.full((h.shape[0],), i * dt, device=device)
        z   = z + dt * model.denoiser.guided(z, h, tau, w)
        record(z)
    return track


def plot_trajectory(split, n_samples=N_SAMPLES, seed=SEED):
    """Усредняет траекторию сэмплинга по первым n_samples примерам сплита."""
    acc, n_seen, n_batches = None, 0, 0
    with seeded(seed, device):
        for batch in loaders[split]:
            if n_seen >= n_samples:
                break
            history, mask, target = (batch[k].to(device) for k in ('history', 'mask', 'target'))
            h  = model.encode_history(history, mask)
            tr = {k: np.array(v) for k, v in
                  sample_with_tracking(h, model.items(target), NUM_STEPS).items()}
            acc = tr if acc is None else {k: acc[k] + tr[k] for k in acc}
            n_seen    += history.shape[0]
            n_batches += 1
    acc   = {k: v / n_batches for k, v in acc.items()}
    steps = np.arange(len(acc['norm_x']))

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    ax = axes[0]
    ax.plot(steps, acc['l2_target'], color='tomato',    lw=2, label='Target item')
    ax.plot(steps, acc['l2_random'], color='steelblue', lw=2, ls='--', label='Random item')
    ax.set_title('L2 Distance'); ax.set_xlabel('Шаг ODE'); ax.set_ylabel('↓ closer')
    ax.legend(fontsize=9); ax.set_xlim(0, len(steps) - 1)

    ax = axes[1]
    ax.plot(steps, acc['norm_x'], color='mediumorchid', lw=2, label='||z_τ||')
    ax.axhline(NOISE_NORM, color='steelblue', ls='--', lw=1.2, label=f'E[||шум||] ≈ {NOISE_NORM:.1f}')
    ax.axhline(TRAIN_TARGET_NORM, color='tomato', ls='--', lw=1.2,
               label=f'E[||e_target||] ≈ {TRAIN_TARGET_NORM:.2f}')
    ax.set_title('Норма ||z_τ||'); ax.set_xlabel('Шаг ODE'); ax.set_ylabel('норма')
    ax.legend(fontsize=9); ax.set_xlim(0, len(steps) - 1); ax.set_ylim(bottom=0)

    ax = axes[2]
    ax.plot(steps, acc['cos_target'], color='tomato',    lw=2, label='Target item')
    ax.plot(steps, acc['cos_random'], color='steelblue', lw=2, ls='--', label='Random item')
    ax.set_title('Cosine Similarity'); ax.set_xlabel('Шаг ODE'); ax.set_ylabel('↑ closer')
    ax.legend(fontsize=9); ax.set_xlim(0, len(steps) - 1)

    plt.suptitle(f'{split.upper()} — DreamRecFMText ODE trajectory  (d_emb={model.d_emb}, '
                 f'{NUM_STEPS} шагов, w_cfg={W_CFG}, {n_seen} примеров)', fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()


plot_trajectory('val')

In [ ]:
plot_trajectory('test')

## 4. Модель против шума и случайного ранжирования

Если ранжировать товары по ненастроенному вектору `N(0, I)` вместо oracle, метрики должны
упасть до случайных. Гистограмма показывает, к какой норме приходит oracle: ориентиры —
средняя норма таргета в train и норма исходного шума.

In [ ]:
# ════════════════════════════════════════════════════════════
# Модель vs шум N(0, I) vs аналитический random
# ════════════════════════════════════════════════════════════
def analytical_random(k):
    return k / N_ITEMS, sum(1.0 / np.log2(r + 2) for r in range(k)) / N_ITEMS

random_row = {}
for k in cfg.eval.top_k:
    random_row[f'HR@{k}'], random_row[f'NDCG@{k}'] = analytical_random(k)

results, oracle_norms = {}, {}
for split in ['val', 'test']:
    r_model, r_noise, targets, norms = [], [], [], []
    with seeded(SEED, device), torch.no_grad():
        for batch in loaders[split]:
            history, mask, target = (batch[k].to(device) for k in ('history', 'mask', 'target'))
            oracle = oracle_for(history, mask)
            r_model.append(retrieve(oracle, W_ITEMS, model.retrieval_metric, MAX_K).cpu())
            r_noise.append(retrieve(torch.randn_like(oracle), W_ITEMS, model.retrieval_metric, MAX_K).cpu())
            norms.append(oracle.norm(dim=-1).cpu())
            targets.append(target.cpu())
    targets = torch.cat(targets)
    oracle_norms[split] = torch.cat(norms).numpy()
    results[split] = pd.DataFrame({
        f'Model ({RETRIEVAL_ROW})': metrics_row(torch.cat(r_model), targets),
        f'Noise ({RETRIEVAL_ROW})': metrics_row(torch.cat(r_noise), targets),
        'Random (analytical)':      random_row,
    }).T[METRICS]

for split, df in results.items():
    print(f'=== {split.upper()} ===')
    print(df.to_string(float_format=lambda x: f'{x:.4f}'))
    print()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (split, color) in zip(axes, [('val', 'steelblue'), ('test', 'tomato')]):
    norms_arr = oracle_norms[split]
    ax.hist(norms_arr, bins=60, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(norms_arr.mean(), color='black', lw=2, label=f'Mean = {norms_arr.mean():.2f}')
    ax.axvline(TRAIN_TARGET_NORM, color='tomato', lw=2, ls='--',
               label=f'E[||e_target||] train = {TRAIN_TARGET_NORM:.2f}')
    ax.axvline(NOISE_NORM, color='steelblue', lw=2, ls='--', label=f'E[||шум||] = {NOISE_NORM:.1f}')
    ax.set_xlabel('||oracle||'); ax.set_ylabel('Количество пользователей')
    ax.set_title(f'{split.upper()} — распределение норм oracle'); ax.legend(fontsize=9)
plt.suptitle('Норма oracle после сэмплинга', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 5. Кривые обучения

`log_train.csv` и `log_val.csv` из папки чекпойнта. Строка `test` в конце `log_val.csv` — это
финальная оценка на best.pt, а не эпоха, поэтому в кривые она не входит.

In [ ]:
# ════════════════════════════════════════════════════════════
# Кривые обучения: train/val loss и val NDCG@20
# ════════════════════════════════════════════════════════════
log_dir   = Path(CKPT_DIR)
train_log = pd.read_csv(log_dir / 'log_train.csv')
val_raw   = pd.read_csv(log_dir / 'log_val.csv')

is_test     = val_raw['epoch'].astype(str) == 'test'
val_log     = val_raw[~is_test].apply(pd.to_numeric, errors='coerce')
test_ndcg20 = float(val_raw.loc[is_test, 'NDCG@20'].iloc[-1]) if is_test.any() else None

best          = val_log.loc[val_log['NDCG@20'].idxmax()]
random_ndcg20 = sum(1.0 / np.log2(r + 2) for r in range(20)) / N_ITEMS

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
ax.plot(train_log['epoch'], train_log['loss'], color='steelblue', lw=1.5, label='Train loss')
ax.plot(val_log['epoch'], val_log['val_loss'], color='tomato', lw=1.5, ls='--', label='Val loss')
ax.axhline(2.0, color='gray', ls=':', lw=1.2, label='E‖u‖²/d = 2 (предсказание нуля)')
ax.set_xlabel('Epoch'); ax.set_ylabel('velocity MSE'); ax.set_title('Train & Val Loss')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(val_log['epoch'], val_log['NDCG@20'], color='mediumorchid', lw=1.5, label='Val NDCG@20')
ax.scatter([best['epoch']], [best['NDCG@20']], color='black', zorder=5, s=60,
           label=f'Best: {best["NDCG@20"]:.4f} @ ep {int(best["epoch"])}')
if test_ndcg20 is not None:
    ax.axhline(test_ndcg20, color='tomato', ls='--', lw=1.2, label=f'Test (тренер): {test_ndcg20:.4f}')
ax.axhline(random_ndcg20, color='gray', ls='--', lw=1.2, label=f'Random = {random_ndcg20:.4f}')
ax.set_xlabel('Epoch'); ax.set_ylabel('NDCG@20'); ax.set_title('Val NDCG@20'); ax.legend(fontsize=8)

plt.suptitle(f'Training curves — {log_dir.name}', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 6. Shuffle-тест: насколько oracle зависит от истории

Oracle строится по истории *другого* пользователя из того же батча. Если метрики падают до
уровня random — модель действительно опирается на историю, а не выдаёт один и тот же вектор.

In [ ]:
# ════════════════════════════════════════════════════════════
# Shuffle-тест
# ════════════════════════════════════════════════════════════
with seeded(SEED, device), torch.no_grad():
    cos_sims, l2_dists = [], []
    for batch in loaders['val']:
        history, mask = batch['history'].to(device), batch['mask'].to(device)
        perm   = torch.randperm(history.shape[0], device=device)
        o_real = oracle_for(history, mask)
        o_shuf = oracle_for(history[perm], mask[perm])
        cos_sims.append(F.cosine_similarity(o_real, o_shuf, dim=-1).cpu())
        l2_dists.append((o_real - o_shuf).norm(dim=-1).cpu())

print(f'Cosine similarity (своя vs чужая история): {torch.cat(cos_sims).mean():.4f}  (1.0 = история не влияет)')
print(f'L2 distance       (своя vs чужая история): {torch.cat(l2_dists).mean():.4f}  (0.0 = история не влияет)')
print(f'L2 между двумя N(0, I):                  ≈ {np.sqrt(2 * model.d_emb):.2f}\n')


@torch.no_grad()
def shuffled_run(split, seed):
    """Метрики, когда oracle строится по истории другого пользователя из батча."""
    ranks, targets = [], []
    with seeded(seed, device):
        for batch in loaders[split]:
            history, mask, target = (batch[k].to(device) for k in ('history', 'mask', 'target'))
            perm   = torch.randperm(history.shape[0], device=device)
            oracle = oracle_for(history[perm], mask[perm])
            ranks.append(retrieve(oracle, W_ITEMS, model.retrieval_metric, MAX_K).cpu())
            targets.append(target.cpu())
    return metrics_row(torch.cat(ranks), torch.cat(targets))


for split, real_runs in RETRIEVAL_RUNS.items():
    runs = [{f'Model ({RETRIEVAL_ROW})': real[RETRIEVAL_ROW],
             'Model, чужая история':     shuffled_run(split, SEED + 100 * i),
             'Random':                   real['Random']}
            for i, real in enumerate(real_runs)]
    print(f'=== {split.upper()} ({N_RUNS} сидов сэмплинга) ===')
    print(mean_std_table(runs).to_string())
    print()

## 7. Диагностика velocity-сети

- **cos(v, u) по τ** — насколько предсказанная скорость совпадает с истинной `u = z₁ − z₀` на
  разных участках траектории. У линейного денойзера коэффициент при `z_τ` один на все τ, а у
  истинного поля он зависит от τ — поэтому cos(v, u) затухает (или меняет знак) к концу.
- **‖t_emb‖** — если выход `time_mlp` почти нулевой, модель не использует время. Для линейного
  денойзера так и должно быть: время входит только слагаемым и может моделировать лишь
  `E[u | τ] = E[z₁] − E[z₀] = 0`.
- **‖W₀‖ по блокам** входа `concat([z_τ, h, t_emb])` — на что опирается денойзер.

In [ ]:
# ════════════════════════════════════════════════════════════
# Диагностика velocity-сети
# ════════════════════════════════════════════════════════════
TAU_GRID = np.linspace(0.0, 0.95, 20)

batch = next(iter(loaders['val']))
history, mask, target = (batch[k].to(device) for k in ('history', 'mask', 'target'))
B, d = history.shape[0], model.d_emb

with seeded(SEED, device), torch.no_grad():
    h      = model.encode_history(history, mask)
    h_shuf = h[torch.randperm(B, device=device)]
    z1     = model.items(target)
    z0     = torch.randn_like(z1)
    u      = z1 - z0

    rows = []
    for tv in TAU_GRID:
        tau = torch.full((B,), float(tv), device=device)
        zt  = (1 - tv) * z0 + tv * z1
        v   = model.denoiser(zt, h, tau)
        v_s = model.denoiser(zt, h_shuf, tau)
        rows.append({
            'tau':               tv,
            'cos(v,u)':          F.cosine_similarity(v, u, dim=-1).mean().item(),
            'cos(v,u), чужая h': F.cosine_similarity(v_s, u, dim=-1).mean().item(),
            '‖v‖/‖u‖':           (v.norm(dim=-1) / u.norm(dim=-1)).mean().item(),
            '‖t_emb‖':           model.denoiser.time_embedding(tau).norm(dim=-1).mean().item(),
        })
diag = pd.DataFrame(rows).set_index('tau')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
ax = axes[0]
ax.plot(diag.index, diag['cos(v,u)'], color='tomato', lw=2, label='своя история')
ax.plot(diag.index, diag['cos(v,u), чужая h'], color='steelblue', lw=2, ls='--', label='чужая история')
ax.axhline(0, color='gray', lw=1)
ax.set_title('cos(v_pred, u_true) по τ'); ax.set_xlabel('τ'); ax.legend(fontsize=9)
ax = axes[1]
ax.plot(diag.index, diag['‖v‖/‖u‖'], color='mediumorchid', lw=2)
ax.set_title('‖v_pred‖ / ‖u_true‖'); ax.set_xlabel('τ')
ax = axes[2]
ax.plot(diag.index, diag['‖t_emb‖'], color='seagreen', lw=2)
ax.set_title('‖t_emb‖ (выход time_mlp)'); ax.set_xlabel('τ'); ax.set_ylim(bottom=0)
plt.suptitle(f'Velocity-сеть (d_emb={d}), val batch', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print(diag.iloc[::4].to_string(float_format=lambda x: f'{x:+.3f}'))

W0 = model.denoiser.linear.weight
print('\n‖W₀‖ по блокам входа:  ' + ' | '.join(
    f'{name} {W0[:, i * d:(i + 1) * d].norm():.2f}' for i, name in enumerate(['z_τ', 'h', 't_emb'])))

## 8. t-SNE и UMAP: oracle против реальных таргет-эмбеддингов (val)

In [ ]:
# ════════════════════════════════════════════════════════════
# t-SNE и UMAP
# ════════════════════════════════════════════════════════════
with seeded(SEED, device), torch.no_grad():
    all_oracles, all_targets = [], []
    for batch in loaders['val']:
        history, mask, target = (batch[k].to(device) for k in ('history', 'mask', 'target'))
        all_oracles.append(oracle_for(history, mask).cpu())
        all_targets.append(model.items(target).cpu())

oracles = torch.cat(all_oracles).numpy()
targets = torch.cat(all_targets).numpy()
N       = len(oracles)
X       = np.concatenate([oracles, targets], axis=0)
labels  = np.array(['Oracle'] * N + ['Target'] * N)
print(f'Fitting on {2 * N} vectors ({N} oracle + {N} target), d={X.shape[1]}')

from sklearn.manifold import TSNE
X_tsne = TSNE(n_components=2, perplexity=40, random_state=SEED, n_jobs=-1).fit_transform(X)
print('t-SNE done')

try:
    import umap
    X_umap   = umap.UMAP(n_components=2, random_state=SEED).fit_transform(X)
    has_umap = True
    print('UMAP done')
except ImportError:
    has_umap = False
    print('UMAP не установлен (pip install umap-learn) — только t-SNE.')

panels = [('t-SNE', X_tsne)] + ([('UMAP', X_umap)] if has_umap else [])
fig, axes = plt.subplots(1, len(panels), figsize=(8 * len(panels), 6), squeeze=False)
style = {'Oracle': ('steelblue', 0.4, 6), 'Target': ('tomato', 0.6, 8)}
for ax, (title, X_2d) in zip(axes[0], panels):
    for label, (color, alpha, size) in style.items():
        m = labels == label
        ax.scatter(X_2d[m, 0], X_2d[m, 1], c=color, alpha=alpha, s=size, label=label, edgecolors='none')
    ax.set_title(f'{title}: Oracle vs Target (val, N={N})', fontsize=12)
    ax.legend(fontsize=10, markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()